# Momentum AI — Model Training Notebook
This notebook trains both ML models required by Momentum AI:
1. **Survival Score Predictor** (XGBoost regression)
2. **RL Scheduler** (PPO via Stable-Baselines3)

After training, download the model files and place them in `momentum-ai/models/`

In [ ]:
# ── Step 1: Install dependencies ──────────────────────────────────────────
!pip install stable-baselines3 gymnasium scikit-learn xgboost pandas numpy matplotlib joblib -q

In [ ]:
# ── Step 2: Mount Google Drive (to save models) ───────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_PATH = '/content/drive/MyDrive/momentum-ai'
os.makedirs(f'{PROJECT_PATH}/models', exist_ok=True)
print('Drive mounted and models folder ready.')

In [ ]:
# ── Step 3: Add project to Python path ───────────────────────────────────
import sys
sys.path.insert(0, PROJECT_PATH)
print('Path set to:', PROJECT_PATH)

## Part 1 — Survival Score Model (XGBoost)

In [ ]:
from ai_engine.survival_score.feature_engineering import generate_synthetic_training_data, FEATURE_COLUMNS
import pandas as pd

# Generate 3000 synthetic training samples
df = generate_synthetic_training_data(3000)
print(f'Generated {len(df)} samples')
print(df.describe())
df.head()

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor
import joblib, numpy as np

X = df[FEATURE_COLUMNS]
y = df['survival_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb', XGBRegressor(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=0
    ))
])

# Cross validation
cv = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
print(f'CV R² scores: {cv}')
print(f'Mean CV R²:   {cv.mean():.4f} ± {cv.std():.4f}')

# Train
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f'\nTest MAE: {mean_absolute_error(y_test, y_pred):.2f}')
print(f'Test R²:  {r2_score(y_test, y_pred):.4f}')

In [ ]:
import matplotlib.pyplot as plt

# Feature importance
importances = model.named_steps['xgb'].feature_importances_
plt.figure(figsize=(8, 5))
plt.barh(FEATURE_COLUMNS, importances, color='steelblue')
plt.xlabel('Importance')
plt.title('Survival Score — Feature Importance')
plt.tight_layout()
plt.savefig(f'{PROJECT_PATH}/models/feature_importance.png', dpi=150)
plt.show()

# Predicted vs Actual
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.4, color='steelblue', s=20)
plt.plot([0, 100], [0, 100], 'r--')
plt.xlabel('Actual Score')
plt.ylabel('Predicted Score')
plt.title('Predicted vs Actual Survival Score')
plt.tight_layout()
plt.savefig(f'{PROJECT_PATH}/models/pred_vs_actual.png', dpi=150)
plt.show()

In [ ]:
# Save survival score model
model_path = f'{PROJECT_PATH}/models/survival_score_model.pkl'
joblib.dump(model, model_path)
print(f'Survival score model saved to: {model_path}')

## Part 2 — RL Scheduler (PPO)

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
from stable_baselines3.common.monitor import Monitor
from ai_engine.rl_scheduler.environment import SchedulingEnv
import numpy as np

def make_env():
    return Monitor(SchedulingEnv(max_tasks=10))

# Vectorized environments for faster training
env = make_vec_env(make_env, n_envs=4)
eval_env = make_vec_env(make_env, n_envs=1)

print('Environments created.')
print('Observation space:', env.observation_space)
print('Action space:     ', env.action_space)

In [ ]:
import os
rl_save_path = f'{PROJECT_PATH}/models'
log_path = f'{PROJECT_PATH}/logs'
os.makedirs(rl_save_path, exist_ok=True)
os.makedirs(log_path, exist_ok=True)

model_rl = PPO(
    policy='MlpPolicy',
    env=env,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    verbose=1,
    tensorboard_log=log_path
)

callbacks = [
    EvalCallback(
        eval_env,
        best_model_save_path=rl_save_path,
        log_path=log_path,
        eval_freq=10_000,
        deterministic=True,
        verbose=1
    ),
    CheckpointCallback(
        save_freq=50_000,
        save_path=rl_save_path,
        name_prefix='momentum_rl_scheduler'
    )
]

print('Training PPO for 300,000 timesteps...')
model_rl.learn(
    total_timesteps=300_000,
    callback=callbacks,
    progress_bar=True
)

final_path = f'{rl_save_path}/momentum_rl_scheduler_final'
model_rl.save(final_path)
print(f'RL model saved to: {final_path}.zip')

In [ ]:
# ── Evaluate the RL agent ─────────────────────────────────────────────────
model_rl = PPO.load(f'{rl_save_path}/momentum_rl_scheduler_final')
test_env = SchedulingEnv(max_tasks=10)

rewards = []
for ep in range(20):
    obs, _ = test_env.reset()
    total_reward = 0
    done = False
    while not done:
        action, _ = model_rl.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = test_env.step(action)
        total_reward += reward
        done = terminated or truncated
    rewards.append(total_reward)

print(f'Mean reward: {np.mean(rewards):.2f} ± {np.std(rewards):.2f}')

# Learning curve
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))
plt.plot(rewards, marker='o', markersize=4)
plt.axhline(np.mean(rewards), color='r', linestyle='--', label=f'Mean: {np.mean(rewards):.2f}')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('RL Scheduler — Evaluation Rewards')
plt.legend()
plt.tight_layout()
plt.savefig(f'{rl_save_path}/rl_eval_rewards.png', dpi=150)
plt.show()

## Done
Download the following files from your Drive and place them in `momentum-ai/models/`:
- `survival_score_model.pkl`
- `momentum_rl_scheduler_final.zip`
- `best_model.zip` (best RL checkpoint)
